In [4]:
!pip install pandas requests sqlalchemy

In [5]:
import pandas as pd

url = "https://datahub.io/core/s-and-p-500-companies/r/constituents.csv"
sp500 = pd.read_csv(url)

sp500.head()

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989


In [6]:
tickers = sp500["Symbol"].tolist()
tickers[:10]

['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A']

In [7]:
API_KEY = "GE64TOKM2QTKCS41"

In [8]:
import requests

all_data = []

for ticker in tickers[:10]:  # start with 10
    url = f"https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={ticker}&apikey={API_KEY}"

    r = requests.get(url)
    data = r.json()

    if "Time Series (Daily)" not in data:
        continue

    for date, values in data["Time Series (Daily)"].items():
        all_data.append([
            ticker,
            date,
            float(values["1. open"]),
            float(values["4. close"])
        ])

In [9]:
df = pd.DataFrame(all_data, columns=["ticker", "date", "open", "close"])

df.head()

,ticker,date,open,close
0,MMM,2026-03-27,143.69,143.04
1,MMM,2026-03-26,147.09,143.99
2,MMM,2026-03-25,148.48,148.05
3,MMM,2026-03-24,145.03,146.67
4,MMM,2026-03-23,144.04,146.56


In [12]:
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["ticker", "date"])

In [13]:
df["daily_return"] = (df["close"] - df["open"]) / df["open"]

In [14]:
df["7_day_avg_close"] = df.groupby("ticker")["close"].transform(lambda x: x.rolling(7).mean())
df["30_day_avg_close"] = df.groupby("ticker")["close"].transform(lambda x: x.rolling(30).mean())

In [15]:
df.to_csv("sp500_etl_dataset.csv", index=False)

In [16]:
from google.colab import files
files.download("sp500_etl_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>